#  Single-Qubit State Tomography
*Reconstructing an unknown quantum state from measurements*

This notebook demonstrates the complete process of single-qubit state tomography:
1. Creating a target quantum state
2. Simulating measurements in different bases
3. Reconstructing the density matrix
4. Visualizing results on the Bloch sphere
5. Analyzing the effect of partial measurements

In [28]:
#imports 
import numpy as np
import matplotlib.pyplot as plt
from qutip import about, basis, sigmax, sigmay, sigmaz, qeye, Qobj, fidelity
from qutip.measurement import measure
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import display, Math, Latex, Markdown, HTML

# Set up for inline plotting
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12

In [34]:
# Create target state |+i⟩ = (|0⟩ + i|1⟩)/√2
target_state = (basis(2, 0) + 1j*basis(2, 1)).unit()
target_dm = target_state * target_state.dag()

print("Target state: |+i⟩ = (|0⟩ + i|1⟩)/√2")
print("\nDensity matrix:")
print(np.round(target_dm.full(), 4))

# Calculate Bloch vector components
x_exp = np.real((target_dm * sigmax()).tr())
y_exp = np.real((target_dm * sigmay()).tr())
z_exp = np.real((target_dm * sigmaz()).tr())

print(f"\nBloch vector components:")
print(f"⟨σ_x⟩ = {x_exp:.4f}")
print(f"⟨σ_y⟩ = {y_exp:.4f}")
print(f"⟨σ_z⟩ = {z_exp:.4f}")

Target state: |+i⟩ = (|0⟩ + i|1⟩)/√2

Density matrix:
[[0.5+0.j  0. -0.5j]
 [0. +0.5j 0.5+0.j ]]

Bloch vector components:
⟨σ_x⟩ = 0.0000
⟨σ_y⟩ = 1.0000
⟨σ_z⟩ = 0.0000


For complete single-qubit tomography, the measured operators must form an operator basis on the Hilbert space of the system, i.e., the measurements must be tomographically complete.

In [35]:
bases = {
    'Z': {
        'operator': sigmaz(),
        'name': 'Z-basis (computational)',
        'states': ['|0⟩', '|1⟩'],
        'eigenstates': [basis(2,0), basis(2,1)],
        'color': 'blue'
    },
    'X': {
        'operator': sigmax(),
        'name': 'X-basis (superposition)',
        'states': ['|+⟩', '|-⟩'],
        'eigenstates': [(basis(2,0)+basis(2,1)).unit(), 
                        (basis(2,0)-basis(2,1)).unit()],
        'color': 'red'
    },
    'Y': {
        'operator': sigmay(),
        'name': 'Y-basis (circular)',
        'states': ['|+i⟩', '|-i⟩'],
        'eigenstates': [(basis(2,0)+1j*basis(2,1)).unit(), 
                        (basis(2,0)-1j*basis(2,1)).unit()],
        'color': 'green'
    }
}

# Display basis information
print("Measurement Bases Defined:")
for basis_name, basis_info in bases.items():
    print(f"\n{basis_name}: {basis_info['name']}")
    print(f"  Operator: {basis_info['operator']}")
    print(f"  Eigenstates: {basis_info['states']}")


Measurement Bases Defined:

Z: Z-basis (computational)
  Operator: Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[ 1.  0.]
 [ 0. -1.]]
  Eigenstates: ['|0⟩', '|1⟩']

X: X-basis (superposition)
  Operator: Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0. 1.]
 [1. 0.]]
  Eigenstates: ['|+⟩', '|-⟩']

Y: Y-basis (circular)
  Operator: Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0.+0.j 0.-1.j]
 [0.+1.j 0.+0.j]]
  Eigenstates: ['|+i⟩', '|-i⟩']


In [49]:
target_state = (basis(2, 0) + 1j*basis(2, 1)).unit()
n_shots = 1000  # Number of measurements per basis
measurement_results = {'Z': [], 'X': [], 'Y': []}
x=0
y=0
z=0

print(f"Simulating {n_shots} measurements in each basis...\n")

for basis_name in ['Z', 'X', 'Y']:
    # Select the measurement operator
    if basis_name == 'Z':
        operator = sigmaz()
    elif basis_name == 'X':
        operator = sigmax()
    else:  # Y
        operator = sigmay()
    
    print(f"\n{basis_name}-basis measurements:")
    
    # Perform n_shots individual measurements
    for shot in range(n_shots):
        # Each measurement needs a fresh copy of the state
        # (In real experiments, each shot uses a new identical copy)
        fresh_state = target_state.copy()
        
        # Measure using QuTiP's measure function
        # measure returns: (collapsed_state, measurement_result)
        # measurement_result is +1 or -1 for Pauli operators
        result, collapsed_state = measure(fresh_state, operator)
        
        # Store the result (+1 or -1)
        measurement_results[basis_name].append(result)
        
    # Calculate statistics
    results_array = np.array(measurement_results[basis_name])
    count_plus = np.sum(results_array == 1)   # +1 outcome
    count_minus = np.sum(results_array == -1)  # -1 outcome

    prob_plus = count_plus / n_shots
    prob_minus = count_minus / n_shots

    if basis_name == 'Z':
        z = 2 * prob_plus - 1
    elif basis_name == 'X':
        x = 2 * prob_plus - 1
    else:  # Y
        y = 2 * prob_plus - 1
    
    print(f"  Counts: +1: {count_plus:5d}, -1: {count_minus:5d}")
    print(f"  Probabilities: P(+1) = {prob_plus:.4f}, P(-1) = {prob_minus:.4f}")
    


Simulating 1000 measurements in each basis...


Z-basis measurements:
  Counts: +1:   498, -1:   502
  Probabilities: P(+1) = 0.4980, P(-1) = 0.5020

X-basis measurements:
  Counts: +1:   519, -1:   481
  Probabilities: P(+1) = 0.5190, P(-1) = 0.4810

Y-basis measurements:
  Counts: +1:  1000, -1:     0
  Probabilities: P(+1) = 1.0000, P(-1) = 0.0000


Using the measurement results, we can reconstruct the density matrix.

In [59]:
#Now reconstructing the density matrix
length = np.sqrt(x**2 + y**2 + z**2)
x=x/length
y=y/length
z=z/length
I = qeye(2)
reconstructed_rho = (I + x*sigmax() + y*sigmay() + z*sigmaz()) / 2
print(reconstructed_rho)


Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0.49800146+0.j        0.01898615-0.4996354j]
 [0.01898615+0.4996354j 0.50199854+0.j       ]]
